In [1]:
import os

In [2]:
%pwd

'f:\\Kidney-Disease-Classification-MLflow-DVC\\research'

In [3]:
os.chdir("../")


In [4]:
%pwd

'f:\\Kidney-Disease-Classification-MLflow-DVC'

In [68]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    

In [ ]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])




        


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config
    

In [ ]:
import os
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size


In [80]:
import py7zr
import os


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        """
        Fetch data from the url
        """
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file

            os.makedirs("artifacts/data_ingestion", exist_ok=True)

            logger.info(
                f"Downloading data from {dataset_url} into file {zip_download_dir}"
            )

            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?export=download&id="

            gdown.download(prefix + file_id, zip_download_dir)

            logger.info(
                f"Downloaded data from {dataset_url} into file {zip_download_dir}"
            )

        except Exception as e:
            raise e

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)

        with py7zr.SevenZipFile(self.config.local_data_file, mode="r") as archive:
            archive.extractall(path=unzip_path)
            

In [81]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e


[2026-08-07 13:58:54,517: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-07 13:58:54,521: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-07 13:58:54,523: INFO: common: created directory at: artifacts]
[2026-08-07 13:58:54,525: INFO: common: created directory at: artifacts/data_ingestion]
[2026-08-07 13:58:54,527: INFO: 1276275552: Downloading data from https://drive.google.com/file/d/1nCwTmYBWTgwxrgEYZ6BPM1L-5ixlRJ99/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?export=download&id=1nCwTmYBWTgwxrgEYZ6BPM1L-5ixlRJ99
From (redirected): https://drive.google.com/uc?export=download&id=1nCwTmYBWTgwxrgEYZ6BPM1L-5ixlRJ99&confirm=t&uuid=9e354136-9593-4485-854b-fa5e2c83b429
To: f:\Kidney-Disease-Classification-MLflow-DVC\artifacts\data_ingestion\data.zip
100%|██████████| 1.58G/1.58G [01:02<00:00, 25.4MB/s]


[2026-08-07 14:00:02,310: INFO: 1276275552: Downloaded data from https://drive.google.com/file/d/1nCwTmYBWTgwxrgEYZ6BPM1L-5ixlRJ99/view?usp=sharing into file artifacts/data_ingestion/data.zip]
